[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-12-materializers.ipynb#scrollTo=kk000001)

---
# Day 12 · Materializers & Data Persistence
**certified-journeys / hamilton-certified** · Day 12 · Production Patterns

> **Goal for today:** Use Hamilton materializers to save pipeline outputs to CSV, Parquet, and JSON — then reload them as inputs to the next pipeline stage.

In [ ]:
%pip install -q sf-hamilton pyarrow

## What are Materializers?

Materializers bridge **pipeline outputs** and **persistent storage** — without touching your function code. They answer the question: *after compute, where does the data go?*

```python
from hamilton.io.materialization import to

dr.materialize(
    to.parquet(id='features', path='/tmp/features.parquet', dependencies=['age_zscore', 'spend_log']),
    to.json(id='metadata', path='/tmp/metadata.json', dependencies=['pipeline_stats']),
    inputs={'raw_df': df}
)
```

| Storage target | Materializer | Best for |
|---|---|---|
| Parquet | `to.parquet` | Feature store, columnar analytics |
| CSV | `to.csv` | Sharing, inspection, small outputs |
| JSON | `to.json` | Metadata, config, small structured data |
| Custom | Implement `DataSaver` | Redis, BigQuery, feature stores |

In [ ]:
import sys, types, tempfile, os
import numpy as np
import pandas as pd
from hamilton import driver
from hamilton.function_modifiers import tag, extract_columns
from hamilton.plugins import h_pandas

# Synthetic dataset
rng = np.random.default_rng(12)
N = 500
raw = pd.DataFrame({
    'customer_id': range(N),
    'age':    rng.integers(18, 75, N).astype(float),
    'spend':  rng.exponential(100, N),
    'tenure': rng.integers(1, 60, N).astype(float),
})

# ── Feature pipeline ───────────────────────────────────────────────────────────
@extract_columns('customer_id', 'age', 'spend', 'tenure')
def raw_df(raw_data: pd.DataFrame) -> pd.DataFrame:
    return raw_data

@tag(feature_type='numerical')
def age_zscore(age: pd.Series) -> pd.Series:
    return (age - age.mean()) / age.std()

@tag(feature_type='numerical')
def spend_log(spend: pd.Series) -> pd.Series:
    return np.log1p(spend)

@tag(feature_type='numerical')
def tenure_years(tenure: pd.Series) -> pd.Series:
    return tenure / 12.0

@tag(feature_type='boolean')
def is_high_spender(spend: pd.Series) -> pd.Series:
    return (spend > spend.quantile(0.75)).astype(float)

@tag(feature_type='numerical')
def engagement_score(spend_log: pd.Series, tenure_years: pd.Series) -> pd.Series:
    return spend_log * (tenure_years / tenure_years.max())

def feature_matrix(
    customer_id: pd.Series,
    age_zscore: pd.Series,
    spend_log: pd.Series,
    tenure_years: pd.Series,
    is_high_spender: pd.Series,
    engagement_score: pd.Series,
) -> pd.DataFrame:
    """Assemble the feature matrix — the main output to persist."""
    return pd.DataFrame({
        'customer_id':    customer_id,
        'age_zscore':     age_zscore,
        'spend_log':      spend_log,
        'tenure_years':   tenure_years,
        'is_high_spender': is_high_spender,
        'engagement_score': engagement_score,
    })

pipeline_module = types.ModuleType('pipeline')
for fn in [raw_df, age_zscore, spend_log, tenure_years,
           is_high_spender, engagement_score, feature_matrix]:
    setattr(pipeline_module, fn.__name__, fn)
sys.modules['pipeline'] = pipeline_module

dr = driver.Builder().with_modules(pipeline_module).build()
print(f'Pipeline ready — {N} customers, 5 features + customer_id')

## Step 1 · Materialize to Parquet

The most important storage format for ML feature pipelines: columnar, compressed, fast to read.

In [ ]:
from hamilton.io.materialization import to

tmp_dir = tempfile.mkdtemp()
parquet_path = os.path.join(tmp_dir, 'features.parquet')

result = dr.materialize(
    to.parquet(
        id='feature_matrix_parquet',
        path=parquet_path,
        dependencies=['feature_matrix'],
        combine=h_pandas.PandasDataFrameResult(),
    ),
    inputs={'raw_data': raw}
)

size_kb = os.path.getsize(parquet_path) / 1024
print(f'Materialized to: {parquet_path}')
print(f'File size: {size_kb:.1f} KB')

# Read back and verify
reloaded = pd.read_parquet(parquet_path)
print(f'Reloaded shape: {reloaded.shape}')
print(f'Columns: {list(reloaded.columns)}')
assert reloaded.shape[0] == N
print('✓ Parquet roundtrip OK')

### What just happened?
- **`dr.materialize(...)`** executes the pipeline and saves the specified node's output to disk in one call.
- `to.parquet(id=..., path=..., dependencies=[...])` — `id` is a label for the materializer, `dependencies` names the node(s) to save.
- `combine=h_pandas.PandasDataFrameResult()` tells the materializer how to assemble multiple requested nodes into a single DataFrame before saving.

## Step 2 · Materialize Multiple Outputs Simultaneously

In [ ]:
csv_path  = os.path.join(tmp_dir, 'features.csv')
parquet2  = os.path.join(tmp_dir, 'features_v2.parquet')

result2 = dr.materialize(
    # Save full feature matrix as Parquet
    to.parquet(
        id='full_features',
        path=parquet2,
        dependencies=['feature_matrix'],
        combine=h_pandas.PandasDataFrameResult(),
    ),
    # Also save as CSV for inspection
    to.csv(
        id='features_csv',
        path=csv_path,
        dependencies=['feature_matrix'],
        combine=h_pandas.PandasDataFrameResult(),
    ),
    inputs={'raw_data': raw}
)

parquet_kb = os.path.getsize(parquet2) / 1024
csv_kb     = os.path.getsize(csv_path) / 1024

print(f'Parquet: {parquet_kb:.1f} KB  |  CSV: {csv_kb:.1f} KB')
print(f'Compression ratio: {csv_kb / parquet_kb:.1f}× (CSV vs Parquet)')

csv_reloaded = pd.read_csv(csv_path)
assert csv_reloaded.shape == (N, 6)
print('✓ Both materializers ran in one dr.materialize() call')

### What just happened?
- **Multiple materializers in one call** — Hamilton computes the pipeline once and fans out to both storage targets.
- Parquet is typically 3–10× smaller than CSV for numerical data.
- The pipeline executed only once — Hamilton's DAG engine doesn't recompute shared nodes.

## Step 3 · Custom DataSaver — Write to an In-Memory Feature Store

In [ ]:
from hamilton.io.data_adapters import DataSaver
from typing import Any, Type

# Simulated in-memory feature store (dict of DataFrames keyed by feature set name)
FEATURE_STORE: dict[str, pd.DataFrame] = {}


class FeatureStoreSaver(DataSaver):
    """Save a DataFrame to the in-memory FEATURE_STORE under a named key."""

    def __init__(self, feature_set_name: str, version: str = 'latest'):
        self.feature_set_name = feature_set_name
        self.version = version

    def save_data(self, data: pd.DataFrame) -> dict[str, Any]:
        key = f'{self.feature_set_name}:{self.version}'
        FEATURE_STORE[key] = data.copy()
        return {
            'feature_set_name': self.feature_set_name,
            'version': self.version,
            'rows': len(data),
            'columns': list(data.columns),
        }

    @classmethod
    def applicable_types(cls) -> list[Type]:
        return [pd.DataFrame]

    @classmethod
    def name(cls) -> str:
        return 'feature_store'


# Wire the custom saver using to.custom
from hamilton.io.materialization import to

result3 = dr.materialize(
    to.custom(
        id='churn_features_v1',
        savers=[FeatureStoreSaver(feature_set_name='churn_features', version='v1')],
        dependencies=['feature_matrix'],
        combine=h_pandas.PandasDataFrameResult(),
    ),
    inputs={'raw_data': raw}
)

print('Feature store contents:', list(FEATURE_STORE.keys()))
stored = FEATURE_STORE['churn_features:v1']
print(f'Stored DataFrame: {stored.shape[0]} rows × {stored.shape[1]} columns')
assert stored.shape == (N, 6)
print('✓ Custom DataSaver wrote to in-memory feature store')

### What just happened?
- **`DataSaver`** is the interface for custom materializers — implement `save_data`, `applicable_types`, and `name`.
- `to.custom(savers=[...])` uses your custom saver.
- In production: replace `FEATURE_STORE` dict with a real feature store client (Feast, Hopsworks, Redis).

## Step 4 · Load Materialized Data as Pipeline Input

In [ ]:
# A second-stage pipeline that reads the saved Parquet
def raw_features(parquet_path: str) -> pd.DataFrame:
    """Load the pre-computed feature matrix from Parquet."""
    return pd.read_parquet(parquet_path)

def high_value_customers(
    raw_features: pd.DataFrame
) -> pd.DataFrame:
    """Filter to high-spender customers for a downstream model."""
    return raw_features[raw_features['is_high_spender'] == 1.0].copy()

def risk_score(raw_features: pd.DataFrame) -> pd.Series:
    """Simple risk score: high z-scored age + low engagement = higher risk."""
    return raw_features['age_zscore'] - raw_features['engagement_score']

scoring_module = types.ModuleType('scoring')
for fn in [raw_features, high_value_customers, risk_score]:
    setattr(scoring_module, fn.__name__, fn)
sys.modules['scoring'] = scoring_module

dr_score = driver.Builder().with_modules(scoring_module).build()
score_result = dr_score.execute(
    ['high_value_customers', 'risk_score'],
    inputs={'parquet_path': parquet_path}  # path from Step 1
)

print(f'High-value customers: {len(score_result["high_value_customers"])} / {N}')
print(f'Risk score range: {score_result["risk_score"].min():.3f} – {score_result["risk_score"].max():.3f}')
print('✓ Second-stage pipeline consumed the materialised Parquet')

### What just happened?
- **Two independent pipelines** — Stage 1 computes and saves; Stage 2 loads and scores.
- Decoupling stages via materialized files lets you run them on different schedules or machines.
- This is the production MLOps pattern: feature pipeline → feature store → scoring pipeline.

## Step 5 · Verify Materializer Metadata

In [ ]:
# Materializers return metadata about what was saved
import json

json_path = os.path.join(tmp_dir, 'run_metadata.json')

def pipeline_metadata(
    feature_matrix: pd.DataFrame
) -> dict:
    """Capture run metadata: row count, feature names, null counts."""
    return {
        'rows': int(len(feature_matrix)),
        'features': list(feature_matrix.columns),
        'null_counts': feature_matrix.isna().sum().to_dict(),
        'dtypes': {c: str(dt) for c, dt in feature_matrix.dtypes.items()},
    }

meta_module = types.ModuleType('metadata')
for fn in [raw_df, age_zscore, spend_log, tenure_years,
           is_high_spender, engagement_score, feature_matrix, pipeline_metadata]:
    setattr(meta_module, fn.__name__, fn)
sys.modules['metadata'] = meta_module

dr_meta = driver.Builder().with_modules(meta_module).build()

result_meta = dr_meta.materialize(
    to.json(
        id='run_metadata',
        path=json_path,
        dependencies=['pipeline_metadata'],
    ),
    inputs={'raw_data': raw}
)

with open(json_path) as f:
    saved_meta = json.load(f)

print('Saved metadata:', json.dumps(saved_meta, indent=2))

# Clean up temp files
import shutil
shutil.rmtree(tmp_dir)
print('\n✓ Metadata JSON materialized and verified')

### What just happened?
- **`to.json`** writes a dict or list to JSON — useful for run metadata, model cards, and data quality reports.
- Capturing null counts and dtypes at pipeline run time creates an audit trail without a separate data quality tool.

In [ ]:
# Challenge: implement a FeatureStoreLoader (DataLoader) that:
# 1. Reads from the FEATURE_STORE dict by key
# 2. Returns a pd.DataFrame
# 3. Can be used with `from_` (Hamilton's load syntax) as a pipeline input
# Hint: DataLoader implements load_data() and applicable_types()

# from hamilton.io.data_adapters import DataLoader
# class FeatureStoreLoader(DataLoader):
#     def __init__(self, key: str): ...
#     def load_data(self, type_: Type) -> tuple[pd.DataFrame, dict]: ...
#     @classmethod
#     def applicable_types(cls): return [pd.DataFrame]
#     @classmethod
#     def name(cls): return 'feature_store'

print('Implement FeatureStoreLoader and use it to load churn_features:v1 as a pipeline input!')

---
## Day 12 key concepts recap

| Concept | What to remember |
|---|---|
| `dr.materialize(to.parquet(...))` | Execute + save in one call |
| Multiple materializers | One pipeline run, N storage targets |
| `DataSaver` interface | Custom storage: `save_data`, `applicable_types`, `name` |
| Two-stage pipelines | Stage 1 materializes → Stage 2 loads via `inputs={'path': ...}` |
| `to.json` for metadata | Cheap audit trail: row counts, null counts, dtypes |

> **Tip:** Parquet is almost always the right format for ML features: typed, compressed, fast columnar reads. Use CSV only for human inspection or tool interoperability.

---
## What's next
**Day 13** → Architecture review and best practices: when to split modules, DAG design smells, and the Hamilton style guide for production teams.

Mark Day 12 complete in your [tracker](../index.html).